In [1]:
!pip install -U bitsandbytes>=0.46.1
# temp fix

zsh:1: 0.46.1 not found


In [2]:
import os
import sys
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Configurazione del percorso e Cache (Colab o Locale)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    
    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    
    # Impostiamo HF_HOME prima di importare hf_hub o transformers
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    
    print(f"Ambiente Colab e Google Drive configurati. Cache modelli in: {hf_cache_dir}")
except ImportError:
    print("Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.")

/home/zizzis/Code/Politecnico/Large_Language_Models/CLARITY/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.


In [3]:
# 2. Gestione del Token di Autenticazione
hf_token = None
try:
    from google.colab import userdata
    print("Acquisizione token da Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Lettura token dal file .env locale...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("Errore: File .env non trovato.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Autenticazione Hugging Face completata.")
else:
    print("Attenzione: Token HF non trovato. Possibili errori nel download del modello.")

Lettura token dal file .env locale...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Autenticazione Hugging Face completata.


## Direct clarity classification (Classificazione diretta della chiarezza)

Questo task consiste nel prevedere direttamente una delle tre macro-categorie di chiarezza (il livello più alto della tassonomia), valutando il numero di interpretazioni che la risposta ammette. Le etichette sono:
   * **Clear Reply** (Risposta chiara): ammette una sola interpretazione.
   * **Ambivalent Reply** (Risposta ambivalente): viene fornita una risposta valida ma strutturata in modo da prestarsi a molteplici interpretazioni.
   * **Clear Non-Reply** (Non-risposta chiara): il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

In [ ]:
# 3. Caricamento del Modello (Quantizzazione 4-bit)
model_id = "meta-llama/Llama-3.1-8B-Instruct" 

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama 3.1 non ha un pad token di default, usiamo l'eos_token per il fine-tuning
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Inizializzazione configurazione BitsAndBytes per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Trasferimento dei pesi del modello in VRAM...")

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16  # Corretto da dtype
)

print("Caricamento completato con successo. Sistema pronto per l'inferenza.")

Inizializzazione configurazione BitsAndBytes per meta-llama/Llama-3.1-8B-Instruct...
Trasferimento dei pesi del modello in VRAM...


Fetching 4 files:   0%|          | 0/4 [04:18<?, ?it/s]


In [ ]:
# 4. Preparazione per PEFT (LoRA / DoRA)
print("Preparazione del modello per il training a 4-bit...")
model = prepare_model_for_kbit_training(model)

# Configurazione DoRA basata sui layer di attenzione di Llama
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True # Attiviamo DoRA come suggerito per performance migliori
)

print("Applicazione degli adattatori LoRA/DoRA...")
model = get_peft_model(model, peft_config)

# Stampa un riepilogo per verificare di star addestrando solo una frazione dei parametri
model.print_trainable_parameters()

In [ ]:
# 5. Caricamento e Formattazione del Dataset Reale (Task 1: Clarity Classification)
import os
from datasets import load_from_disk

# Percorso della cartella che contiene i file .arrow, dataset_info.json, ecc.
# Assicurati che questo path sia corretto rispetto a dove hai estratto i dati
dataset_path = "dataset/QEvasion/train" 

print(f"Caricamento del dataset da: {dataset_path}...")
try:
    # Carica il dataset salvato in formato arrow/directory di Hugging Face
    dataset = load_from_disk(dataset_path)
    print(f"Dataset caricato con successo! Numero di esempi: {len(dataset)}")
except Exception as e:
    print(f"Errore nel caricamento del dataset. Verifica il percorso. Dettaglio: {e}")

# Definiamo il System Prompt specifico per il Task 1
system_prompt_task1 = """Sei un esperto analista di discorsi politici. Il tuo compito è classificare la risposta di un politico a una domanda diretta in una delle seguenti 3 categorie:
1. 'Clear Reply': La risposta ammette una sola interpretazione chiara.
2. 'Ambivalent Reply': Viene fornita una risposta, ma è strutturata in modo da prestarsi a molteplici interpretazioni o essere vaga.
3. 'Clear Non-Reply': Il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

Rispondi SOLO con il nome esatto della categoria (Clear Reply, Ambivalent Reply o Clear Non-Reply), senza aggiungere altro testo."""

def format_prompt_task1(example):
    """
    Formatta ogni riga del dataset nel formato chat richiesto da Llama 3.1,
    utilizzando le colonne reali del dataset QEvasion.
    """
    # Usiamo le chiavi estratte dal tuo file dataset_info.json
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    label = example.get('clarity_label', '')
    
    user_message = f"Domanda: {domanda}\nRisposta del politico: {risposta}"
    
    # Strutturiamo i messaggi
    messages = [
        {"role": "system", "content": system_prompt_task1},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": str(label)}
    ]
    
    # Applichiamo il template nativo del tokenizer di Llama
    testo_formattato = tokenizer.apply_chat_template(messages, tokenize=False)
    
    return {"text": testo_formattato}

print("Formattazione del dataset in corso...")
# Applichiamo la funzione a tutto il dataset
# Rimuoviamo le vecchie colonne per alleggerire la memoria, tenendo solo la colonna 'text' utile al training
formatted_dataset = dataset.map(
    format_prompt_task1, 
    remove_columns=dataset.column_names 
)

# Stampiamo un esempio per verificare che il template sia applicato correttamente
print("\n--- Esempio di Prompt Formattato per Llama 3.1 ---")
print(formatted_dataset[0]["text"])

In [ ]:
# 6. Configurazione SFTTrainer e Avvio Addestramento
from trl import SFTTrainer
from transformers import TrainingArguments

# Creiamo una cartella per i risultati su Google Drive (o in locale)
output_model_dir = "./clarity_task1"
os.makedirs(output_model_dir, exist_ok=True)

print("Configurazione degli iperparametri di addestramento...")

training_arguments = TrainingArguments(
    output_dir=output_model_dir,
    per_device_train_batch_size=2,          # Batch size per GPU (tienilo basso, 2 o 4)
    gradient_accumulation_steps=4,          # Accumula i gradienti per simulare un batch size maggiore (2x4=8)
    optim="paged_adamw_32bit",              # Ottimizzatore efficiente per QLoRA
    save_steps=50,                          # Salva un checkpoint ogni 50 step
    logging_steps=10,                       # Stampa i log ogni 10 step
    learning_rate=2e-4,                     # Learning rate standard per LoRA
    weight_decay=0.001,
    fp16=True,                              # Usa precisione mista a 16-bit
    bf16=False,                             # Metti True SOLO se usi GPU Ampere (es. A100 su Colab Pro)
    max_grad_norm=0.3,                      # Previene l'esplosione dei gradienti
    max_steps=100,                          # NUMERO DI STEP PER TEST. Rimuovi o commenta per addestrare su tutte le epoche
    # num_train_epochs=1,                   # Usa questo invece di max_steps per fare epoche complete
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine"
)

print("Inizializzazione del SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",              # La colonna che abbiamo creato nella Cella 5
    max_seq_length=512,                     # Lunghezza massima del prompt (aumenta a 1024 se le risposte sono molto lunghe)
    tokenizer=tokenizer,
    args=training_arguments,
)

# Svuota la cache di PyTorch prima di iniziare
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Inizio dell'addestramento! 🚀")
trainer.train()

# Salvataggio del modello finale (Adattatori LoRA/DoRA)
final_save_path = f"{output_model_dir}/modello_finale"
trainer.model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"Addestramento completato! Adattatori salvati in: {final_save_path}")

In [ ]:
# 7. Test e Inferenza sul Dataset di Test
import torch
import os
from datasets import load_from_disk
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("--- FASE DI TEST ---")

# 1. Caricamento o verifica del modello
# Se stiamo eseguendo questa cella in un momento diverso e il modello non è in memoria, lo ricarichiamo
if 'model' not in locals():
    print("Modello non trovato in memoria. Inizializzo il caricamento dal Drive/Locale...")
    
    # Percorsi
    model_id = "meta-llama/Llama-3.1-8B-Instruct"
    lora_path = "./risultati_clarity_task1/modello_finale"
    
    tokenizer = AutoTokenizer.from_pretrained(lora_path) # Carichiamo il tokenizer salvato
    
    # Ri-configuriamo la quantizzazione
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    # Carichiamo il modello base a 4-bit
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_config,
        torch_dtype=torch.float16
    )
    
    # Fondiamo il modello base con i nostri adattatori addestrati
    print(f"Caricamento adattatori LoRA da: {lora_path}")
    model = PeftModel.from_pretrained(base_model, lora_path)
    print("Modello caricato con successo!")
else:
    print("Modello già presente in memoria. Procedo con l'inferenza usando il modello corrente.")

model.eval() # Impostiamo il modello in modalità valutazione

# 2. Caricamento del dataset di Test
test_dataset_path = "dataset/QEvasion/test"
print(f"\nCaricamento dataset di test da: {test_dataset_path}...")
try:
    test_dataset = load_from_disk(test_dataset_path)
    print(f"Dataset di test caricato! Numero totale di esempi: {len(test_dataset)}")
except Exception as e:
    print(f"Errore nel caricamento del dataset di test: {e}")

# Ri-definiamo il system prompt (se non fosse in memoria)
system_prompt_task1 = """Sei un esperto analista di discorsi politici. Il tuo compito è classificare la risposta di un politico a una domanda diretta in una delle seguenti 3 categorie:
1. 'Clear Reply': La risposta ammette una sola interpretazione chiara.
2. 'Ambivalent Reply': Viene fornita una risposta, ma è strutturata in modo da prestarsi a molteplici interpretazioni o essere vaga.
3. 'Clear Non-Reply': Il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

Rispondi SOLO con il nome esatto della categoria (Clear Reply, Ambivalent Reply o Clear Non-Reply), senza aggiungere altro testo."""

# 3. Funzione di Inferenza
def predici_classe(example):
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    
    messages = [
        {"role": "system", "content": system_prompt_task1},
        {"role": "user", "content": f"Domanda: {domanda}\nRisposta del politico: {risposta}"}
    ]
    
    # add_generation_prompt=True aggiunge <|start_header_id|>assistant<|end_header_id|> per innescare la generazione
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Tokenizziamo
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generazione
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,      # Pochi token: vogliamo solo il nome dell'etichetta
            temperature=0.0,        # Temperature a 0 per avere l'output più probabile e deterministico (zero creatività)
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decodifichiamo SOLO i token generati dal modello (ignoriamo il prompt)
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    predizione = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    return predizione

# 4. Esecuzione del test sui primi 5 esempi
print("\n--- INIZIO VALUTAZIONE SUI PRIMI 5 ESEMPI ---")
num_test = min(5, len(test_dataset))

esatti = 0
for i in range(num_test):
    esempio = test_dataset[i]
    vero_label = str(esempio.get('clarity_label', '')).strip()
    
    predizione = predici_classe(esempio)
    
    corretto = "✅" if predizione.lower() == vero_label.lower() else "❌"
    if predizione.lower() == vero_label.lower():
        esatti += 1
        
    print(f"\nEsempio {i+1}:")
    print(f"Vera Etichetta  : {vero_label}")
    print(f"Predizione LLM  : {predizione} {corretto}")

print(f"\nAccuratezza sui primi {num_test} esempi: {esatti}/{num_test} ({(esatti/num_test)*100:.1f}%)")

## Evasion-based clarity classification (Classificazione basata sull'evasione)
Si tratta di un task più a grana fine (il livello inferiore della tassonomia) che prevede un approccio in due fasi. Il modello deve prima prevedere l'esatta tecnica di evasione utilizzata scegliendo tra **9 sotto-categorie** specifiche (ad esempio *Implicit*, *Dodging*, *Deflection*, *General*, *Claims ignorance*, ecc.). Una volta individuata la tecnica di evasione, il modello deduce la categoria di chiarezza principale (delle 3 viste sopra) risalendo la gerarchia della tassonomia.